In [20]:
!pip install streamlit plotly pyngrok -q
print("✓ Dependencies installed")

✓ Dependencies installed


In [21]:
import streamlit as st
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import time
from openai import AzureOpenAI  # Penting untuk fitur AI rekomendasi

In [22]:
import os

if not os.path.exists('.streamlit'):
    os.makedirs('.streamlit')

with open('.streamlit/secrets.toml', 'w') as f:
    f.write('AZURE_KEY = "4ji4iP9qKL06gR2Z3lSEuSE8jm7KOqFRgu7vlHtEdJrHUwCfOXtsJQQJ99CDACNns7RXJ3w3AAABACOGPn3O"\n')
    f.write('AZURE_ENDPOINT = "https://projekjudol.openai.azure.com/"\n')
    f.write('AZURE_DEPLOY = "gpt-4o"\n')

In [23]:
DASHBOARD_CODE = '''

import os
import streamlit as st
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import time
from openai import AzureOpenAI  # Penting untuk fitur AI rekomendasi

st.set_page_config(
    page_title="JudolGuard",
    page_icon="🛡️",
    layout="wide",
    initial_sidebar_state="expanded"
)

st.markdown("""
<style>
@import url('https://fonts.googleapis.com/css2?family=DM+Sans:wght@300;400;500;600&display=swap');
html, body, [class*="css"] { font-family: "DM Sans", sans-serif; }
.metric-card {
    background: #1a1d27; border: 1px solid #2d3142;
    border-radius: 12px; padding: 1.2rem 1.5rem;
    text-align: center; margin-bottom: 8px;
}
.metric-label { font-size: 12px; color: #6b7280; letter-spacing:.08em; text-transform:uppercase; margin-bottom:.4rem; }
.metric-value { font-size: 30px; font-weight: 600; line-height: 1.1; }
.metric-sub   { font-size: 12px; color: #6b7280; margin-top:.2rem; }
.badge-low      { background:#064e3b;color:#6ee7b7;padding:3px 10px;border-radius:20px;font-size:12px;font-weight:500; }
.badge-medium   { background:#78350f;color:#fcd34d;padding:3px 10px;border-radius:20px;font-size:12px;font-weight:500; }
.badge-high     { background:#7c2d12;color:#fb923c;padding:3px 10px;border-radius:20px;font-size:12px;font-weight:500; }
.badge-critical { background:#450a0a;color:#f87171;padding:3px 10px;border-radius:20px;font-size:12px;font-weight:500; }
.explanation-box {
    background: #1a1d27; border-left: 3px solid #6366f1;
    border-radius: 0 8px 8px 0; padding: 1rem 1.2rem;
    font-size: 14px; line-height: 1.7; color: #d1d5db; margin:.8rem 0;
}
.section-title {
    font-size:12px;font-weight:500;color:#6b7280;
    letter-spacing:.1em;text-transform:uppercase;
    margin-bottom:.8rem;margin-top:1.2rem;
}
.shift-row {
    background:#1a1d27; border:1px solid #2d3142; border-radius:8px;
    padding:10px 14px; margin-bottom:6px;
}
#MainMenu{visibility:hidden;}footer{visibility:hidden;}header{visibility:hidden;}
</style>
""", unsafe_allow_html=True)

LEVEL_COLORS = {
    "Low"     : "#6ee7b7",
    "Medium"  : "#fcd34d",
    "High"    : "#fb923c",
    "Critical": "#f87171"
}
PROFILE_COLORS = {
    "normal"       : "#60a5fa",
    "early_stage"  : "#fcd34d",
    "escalating"   : "#fb923c",
    "heavy_gambler": "#f87171"
}
PROFILE_FILL = {
    "normal"       : "rgba(96,165,250,0.15)",
    "early_stage"  : "rgba(252,211,77,0.15)",
    "escalating"   : "rgba(251,146,60,0.15)",
    "heavy_gambler": "rgba(248,113,113,0.15)"
}
REC_DETAIL = {
    "Low"     : ("✅ Monitor Pasif",    "Tidak ada tindakan segera.",                              "#064e3b","#6ee7b7"),
    "Medium"  : ("📢 Kirim Notifikasi", "Kirim pesan edukasi risiko judol ke nasabah.",            "#78350f","#fcd34d"),
    "High"    : ("🚫 Batasi Transfer",  "Terapkan batas harian Rp500.000 + konfirmasi biometrik.","#7c2d12","#fb923c"),
    "Critical": ("🔴 Eskalasi Segera",  "Freeze transaksi + laporkan ke tim compliance & OJK.",   "#450a0a","#f87171"),
}

# ════════════════════════════════════════════════════════════
# SECRETS — aman tanpa secrets.toml (priority: secrets → env → default)
# ════════════════════════════════════════════════════════════
def _get_secret(key, default=""):
    try:
        return st.secrets[key]
    except Exception:
        return os.environ.get(key, default)

AZURE_KEY      = _get_secret("AZURE_KEY")
AZURE_ENDPOINT = _get_secret("AZURE_ENDPOINT")
AZURE_DEPLOY   = _get_secret("AZURE_DEPLOY", "gpt-4o")

# ════════════════════════════════════════════════════════════
# AZURE OPENAI
# ════════════════════════════════════════════════════════════
def get_azure_recommendation(acc_row):
    if not AZURE_KEY or not AZURE_ENDPOINT:
        return None
    try:
        from openai import AzureOpenAI
        client = AzureOpenAI(api_key=AZURE_KEY, api_version="2024-02-01", azure_endpoint=AZURE_ENDPOINT)
        prompt = f"""Kamu adalah sistem AI compliance untuk e-wallet Indonesia yang mendeteksi judi online.
Berikan analisis SINGKAT dan ACTIONABLE (3-4 kalimat) untuk akun berikut:

Data perilaku:
- Risk Score    : {acc_row["final_risk_score"]:.1f}/100
- Risk Level    : {acc_row["risk_level"]}
- Night ratio   : {acc_row["avg_night_ratio"]:.1%}
- Frekuensi     : {acc_row["avg_tx_24h"]:.1f} transaksi/24 jam
- Penerima unik : {acc_row["avg_unique_recv"]:.1f} akun/7 hari
- Temporal shift: {acc_row["avg_temporal_shift"]:+.3f}
- QRIS ratio    : {acc_row["avg_qris_ratio"]:.1%}
- Burst score   : {acc_row["avg_burst_score"]:.2f}x

Format jawaban:
[ANALISIS] Jelaskan pola perilaku yang terdeteksi.
[INDIKATOR UTAMA] Sebutkan 2 indikator paling mencurigakan dengan angkanya.
[TINDAKAN] Rekomendasikan 1 tindakan konkret yang harus segera dilakukan tim compliance.

Gunakan bahasa Indonesia yang profesional dan ringkas."""
        resp = client.chat.completions.create(
            model=AZURE_DEPLOY,
            messages=[
                {"role": "system", "content": "Kamu analis risiko keuangan digital. Singkat, akurat, actionable."},
                {"role": "user",   "content": prompt}
            ],
            temperature=0.3, max_tokens=250
        )
        return resp.choices[0].message.content.strip()
    except Exception as e:
        return f"Error koneksi Azure OpenAI: {e}"

# ════════════════════════════════════════════════════════════
# LOAD DATA
# ════════════════════════════════════════════════════════════
@st.cache_data
def load_data():
    risk = None # Initialize risk
    features = None # Initialize features

    try:
        risk = pd.read_csv("data/risk_scores_with_explanation.csv")
    except FileNotFoundError:
        try:
            risk = pd.read_csv("data/risk_scores.csv")
            risk["explanation"] = "Klik Generate AI Report untuk analisis."
        except FileNotFoundError:
            st.error("File data/risk_scores.csv tidak ditemukan.")
            risk = pd.DataFrame() # Assign an empty DataFrame to prevent UnboundLocalError
            st.stop()
    try:
        features = pd.read_csv("data/judolguard_features.csv")
    except FileNotFoundError:
        try:
            features = pd.read_csv("data/judolguard_.csv")
        except FileNotFoundError:
            features = None
    return risk, features

risk_df, features_df = load_data()

# --- Add a check for empty risk_df and relevant columns after loading data ---
if risk_df.empty or "risk_level" not in risk_df.columns or "profile" not in risk_df.columns:
    st.warning("Risk data is missing or incomplete. Some features will not be available.")
    # If essential columns are missing, create an empty DataFrame to avoid KeyErrors
    risk_df = pd.DataFrame(columns=["account_id", "risk_level", "profile", "final_risk_score", "explanation"])
    filtered = pd.DataFrame()
    # No need to st.stop() here as we've provided an empty DataFrame to proceed gracefully.
else:
    # ════════════════════════════════════════════════════════════
    # SIDEBAR
    # ════════════════════════════════════════════════════════════
    with st.sidebar:
        st.markdown("## 🛡️ JudolGuard")
        st.markdown("<p style='font-size:13px;color:#6b7280;margin-top:-8px;'>Early Behavioral Shift Detection<br>untuk E-Wallet Indonesia</p>", unsafe_allow_html=True)
        st.divider()

        page = st.radio("Nav", ["📊 Overview", "📋 Risk Table", "🔍 Detail Akun"], label_visibility="collapsed")
        st.divider()

        st.markdown("**Filter**")
        sel_levels   = st.multiselect("Level",  ["Critical","High","Medium","Low"], default=["Critical","High","Medium","Low"])

        # Safely get unique profiles, provide empty list if 'profile' column is missing or df is empty
        if not risk_df.empty and "profile" in risk_df.columns:
            available_profiles = sorted(risk_df["profile"].unique())
        else:
            available_profiles = []
        sel_profiles = st.multiselect("Profil", available_profiles, default=available_profiles)
        st.divider()

        st.markdown("**Model Performance**")
        try:
            for line in open("data/model_metrics.txt").read().strip().split("\n"):
                if ":" in line:
                    k, v = line.split(":", 1)
                    st.markdown(f"<div style='display:flex;justify-content:space-between;font-size:13px;margin-bottom:4px;'><span style='color:#6b7280'>{k.strip()}</span><span style='color:#a5b4fc;font-weight:500'>{v.strip()}</span></div>", unsafe_allow_html=True)
        except:
            st.caption("model_metrics.txt tidak ditemukan")

        st.divider()
        if AZURE_KEY:
            st.markdown("<div style='font-size:12px;color:#6ee7b7'>☁️ Azure OpenAI: Terhubung</div>", unsafe_allow_html=True)
        else:
            st.markdown("<div style='font-size:12px;color:#f87171'>☁️ Azure OpenAI: Offline (mode statis)</div>", unsafe_allow_html=True)
        st.divider()
        st.caption("Microsoft Azure AI Impact Challenge 2025")

    filtered = risk_df[
        risk_df["risk_level"].isin(sel_levels) &
        risk_df["profile"].isin(sel_profiles)
    ].copy()

    # ════════════════════════════════════════════════════════════
    # HALAMAN 1 — OVERVIEW
    # ════════════════════════════════════════════════════════════
    if page == "📊 Overview":
        st.markdown("# 📊 Overview Dashboard")
        st.markdown("<p style='color:#6b7280;margin-top:-12px;font-size:14px;'>Sistem deteksi dini perubahan perilaku transaksi — tim compliance e-wallet</p>", unsafe_allow_html=True)
        st.divider()

        total      = len(risk_df)
        n_critical = (risk_df["risk_level"] == "Critical").sum()
        n_high     = (risk_df["risk_level"] == "High").sum()
        n_medium   = (risk_df["risk_level"] == "Medium").sum()
        det_rate   = (n_critical + n_high) / total * 100

        c1, c2, c3, c4, c5 = st.columns(5)
        for col, label, val, color, sub in [
            (c1, "Total Akun",   total,              "#a5b4fc", "dianalisis"),
            (c2, "🔴 Critical",  n_critical,         "#f87171", "eskalasi OJK"),
            (c3, "🟠 High",      n_high,             "#fb923c", "batasi transfer"),
            (c4, "🟡 Medium",    n_medium,           "#fcd34d", "notifikasi"),
            (c5, "Detection %",  f"{det_rate:.1f}%", "#34d399", "High+Critical"),
        ]:
            with col:
                st.markdown(f"""<div class="metric-card">
                    <div class="metric-label">{label}</div>
                    <div class="metric-value" style="color:{color}">{val}</div>
                    <div class="metric-sub">{sub}</div>
                </div>""", unsafe_allow_html=True)

        st.markdown("<br>", unsafe_allow_html=True)
        cl, cr = st.columns(2)

        with cl:
            st.markdown("<div class='section-title'>Distribusi Risk Level</div>", unsafe_allow_html=True)
            lc    = risk_df["risk_level"].value_counts()
            order = ["Critical", "High", "Medium", "Low"]
            fig   = go.Figure(go.Pie(
                labels=order, values=[lc.get(l, 0) for l in order],
                marker_colors=[LEVEL_COLORS[l] for l in order],
                hole=0.55, textinfo="label+percent",
                textfont=dict(size=12, color="white")
            ))
            fig.add_annotation(text=f"<b>{total}</b><br>akun", x=0.5, y=0.5, font_size=14, font_color="white", showarrow=False)
            fig.update_layout(paper_bgcolor="rgba(0,0,0,0)", plot_bgcolor="rgba(0,0,0,0)",
                              showlegend=False, height=280, margin=dict(t=10,b=10,l=10,r=10))
            st.plotly_chart(fig, use_container_width=True)

        with cr:
            st.markdown("<div class='section-title'>Risk Score per Profil</div>", unsafe_allow_html=True)
            fig2 = go.Figure()
            for p in ["normal","early_stage","escalating","heavy_gambler"]:
                d = risk_df[risk_df["profile"] == p]["final_risk_score"]
                if len(d):
                    fig2.add_trace(go.Box(y=d, name=p, marker_color=PROFILE_COLORS[p],
                                          line_color=PROFILE_COLORS[p], fillcolor=PROFILE_FILL[p]))
            fig2.update_layout(paper_bgcolor="rgba(0,0,0,0)", plot_bgcolor="rgba(0,0,0,0)", height=280,
                               margin=dict(t=10,b=10,l=10,r=10), font=dict(color="#9ca3af", size=11),
                               xaxis=dict(gridcolor="#2d3142"),
                               yaxis=dict(gridcolor="#2d3142", title="Risk Score"), showlegend=False)
            st.plotly_chart(fig2, use_container_width=True)

        ca, cb = st.columns(2)
        with ca:
            st.markdown("<div class='section-title'>Temporal Shift Score per Profil</div>", unsafe_allow_html=True)
            if "avg_temporal_shift" in risk_df.columns:
                sm = risk_df.groupby("profile")["avg_temporal_shift"].mean().reset_index()
                sm.columns = ["profile","shift"]
                sm["color"] = sm["shift"].apply(lambda x: "#f87171" if x > 0.01 else "#6ee7b7")
                fig3 = px.bar(sm, x="profile", y="shift", color="color",
                              color_discrete_map="identity", text=sm["shift"].round(3))
                fig3.add_hline(y=0, line_color="#6b7280", line_dash="dash")
                fig3.update_layout(paper_bgcolor="rgba(0,0,0,0)", plot_bgcolor="rgba(0,0,0,0)", height=260,
                                   margin=dict(t=10,b=10,l=10,r=10), font=dict(color="#9ca3af",size=11),
                                   showlegend=False, xaxis=dict(gridcolor="#2d3142"),
                                   yaxis=dict(gridcolor="#2d3142", title="Shift Score"))
                fig3.update_traces(textposition="outside", textfont_color="white")
                st.plotly_chart(fig3, use_container_width=True)

        with cb:
            st.markdown("<div class='section-title'>Night Ratio per Profil</div>", unsafe_allow_html=True)
            if "avg_night_ratio" in risk_df.columns:
                nm = risk_df.groupby("profile")["avg_night_ratio"].mean().reset_index()
                nm.columns = ["profile","night_ratio"]
                fig4 = px.bar(nm, x="profile", y="night_ratio", color="night_ratio",
                              color_continuous_scale=["#6ee7b7","#fcd34d","#f87171"],
                              text=nm["night_ratio"].apply(lambda x: f"{x:.2%}"))
                fig4.update_layout(paper_bgcolor="rgba(0,0,0,0)", plot_bgcolor="rgba(0,0,0,0)", height=260,
                                   margin=dict(t=10,b=10,l=10,r=10), font=dict(color="#9ca3af",size=11),
                                   showlegend=False, coloraxis_showscale=False,
                                   xaxis=dict(gridcolor="#2d3142"), yaxis=dict(gridcolor="#2d3142"))
                fig4.update_traces(textposition="outside", textfont_color="white")
                st.plotly_chart(fig4, use_container_width=True)

        # ════════════════════════════════════════════════════════
        # [BARU] TOP 10 AKUN — PERUBAHAN POLA TERBARU
        # Kriteria: temporal_shift tertinggi (positif = bergeser ke malam)
        # ════════════════════════════════════════════════════════
        st.divider()
        st.markdown("<div class='section-title'>🚨 Top 10 Akun — Perubahan Pola Transaksi Terbaru</div>", unsafe_allow_html=True)
        st.markdown("<p style='font-size:12px;color:#6b7280;margin-top:-8px;margin-bottom:12px;'>Diurutkan berdasarkan Temporal Shift Score tertinggi — akun yang paling cepat berubah ke pola berisiko</p>", unsafe_allow_html=True)

        if "avg_temporal_shift" in risk_df.columns:
            # Ambil top 10 berdasarkan temporal shift positif tertinggi
            top10 = (
                risk_df[risk_df["avg_temporal_shift"] > 0]
                .sort_values("avg_temporal_shift", ascending=False)
                .head(10)
                .reset_index(drop=True)
            )

            # Header tabel
            th1, th2, th3, th4, th5, th6 = st.columns([2.2, 1.3, 1.3, 1.5, 1.5, 1.5])
            for col, label in zip([th1,th2,th3,th4,th5,th6],
                                   ["Account ID", "Level", "Risk Score", "Shift Score", "Night Ratio", "Burst Score"]):
                with col:
                    st.markdown(f"<div style='font-size:11px;color:#6b7280;font-weight:500;letter-spacing:.08em;text-transform:uppercase;padding:4px 0;border-bottom:1px solid #2d3142;margin-bottom:4px;'>{label}</div>", unsafe_allow_html=True)

            for rank, (_, row) in enumerate(top10.iterrows(), start=1):
                level  = row["risk_level"]
                color  = LEVEL_COLORS.get(level, "#6b7280")
                shift  = row["avg_temporal_shift"]
                score  = row["final_risk_score"]
                night  = row.get("avg_night_ratio", 0)
                burst  = row.get("avg_burst_score", 0)

                # Warna shift bar berdasarkan magnitude
                shift_pct   = min(int(abs(shift) * 300), 100)
                shift_color = "#f87171" if shift > 0.05 else "#fcd34d" if shift > 0.02 else "#fb923c"

                c1, c2, c3, c4, c5, c6 = st.columns([2.2, 1.3, 1.3, 1.5, 1.5, 1.5])
                with c1:
                    st.markdown(
                        f"<div style='padding:8px 0;'>"
                        f"<span style='font-size:11px;color:#4b5563;font-weight:600;margin-right:6px;'>#{rank}</span>"
                        f"<span style='font-size:13px;font-weight:500;color:#e2e8f0;'>{row['account_id']}</span>"
                        f"</div>", unsafe_allow_html=True
                    )
                with c2:
                    st.markdown(f"<div style='padding:8px 0;'><span class='badge-{level.lower()}'>{level}</span></div>", unsafe_allow_html=True)
                with c3:
                    st.markdown(f"<div style='padding:8px 0;font-size:13px;font-weight:500;color:{color};'>{score:.1f}/100</div>", unsafe_allow_html=True)
                with c4:
                    st.markdown(
                        f"<div style='padding:8px 0;'>"
                        f"<div style='font-size:13px;font-weight:600;color:{shift_color};'>+{shift:.4f}</div>"
                        f"<div style='background:#2d3142;border-radius:4px;height:3px;margin-top:4px;'>"
                        f"<div style='background:{shift_color};width:{shift_pct}%;height:3px;border-radius:4px;'></div></div>"
                        f"</div>", unsafe_allow_html=True
                    )
                with c5:
                    st.markdown(f"<div style='padding:8px 0;font-size:13px;color:#9ca3af;'>{night:.1%}</div>", unsafe_allow_html=True)
                with c6:
                    st.markdown(f"<div style='padding:8px 0;font-size:13px;color:#9ca3af;'>{burst:.2f}x</div>", unsafe_allow_html=True)

                st.markdown("<hr style='margin:0;border:none;border-top:0.5px solid #2d3142;'>", unsafe_allow_html=True)

            if top10.empty:
                st.caption("Tidak ada akun dengan temporal shift positif yang terdeteksi.")
        else:
            st.caption("Kolom avg_temporal_shift tidak tersedia di dataset.")

        st.divider()
        st.markdown("<div class='section-title'>Azure AI Stack</div>", unsafe_allow_html=True)
        ca2, cb2, cc2 = st.columns(3)
        for col, title, items in [
            (ca2, "☁️ Azure OpenAI (GPT-4o)",    ["Synthetic data generation","Risk explanation per akun","Dynamic recommendation (real-time)"]),
            (cb2, "🤖 Azure ML Registry",          ["Model: JudolGuard-Behavior-Model v1","Experiment tracking (MLflow)","Workspace: ML_JudolGuard"]),
            (cc2, "🔬 Isolation Forest Pipeline",  ["Anomaly detection layer","XGBoost risk classifier","PR-AUC: 0.9655 | F1: 0.8598"]),
        ]:
            with col:
                items_html = "".join([f"✓ {i}<br>" for i in items])
                st.markdown(f"""<div class="metric-card" style="text-align:left">
                    <div style="color:#a5b4fc;font-weight:500;margin-bottom:6px">{title}</div>
                    <div style="font-size:12px;color:#6b7280;line-height:1.8">{items_html}</div>
                </div>""", unsafe_allow_html=True)


    # ════════════════════════════════════════════════════════════
    # HALAMAN 2 — RISK TABLE  [PAGINATION: 20/page + nomor halaman]
    # ════════════════════════════════════════════════════════════
    elif page == "📋 Risk Table":
        st.markdown("# 📋 Risk Table")
        st.markdown("<p style='color:#6b7280;margin-top:-12px;font-size:14px;'>Diurutkan berdasarkan risk score tertinggi</p>", unsafe_allow_html=True)

        search = st.text_input("🔍 Cari Account ID", placeholder="Ketik account ID...")
        if search:
            filtered = filtered[filtered["account_id"].str.contains(search, case=False, na=False)]

        st.divider()

        table      = filtered.sort_values("final_risk_score", ascending=False).reset_index(drop=True)
        PAGE_SIZE  = 20
        total_rows = len(table)
        total_pages = max(1, int(np.ceil(total_rows / PAGE_SIZE)))

        # Session state untuk halaman aktif
        if "risk_page" not in st.session_state:
            st.session_state["risk_page"] = 1
        # Reset ke hal 1 saat search berubah
        if search:
            st.session_state["risk_page"] = 1

        current_page = st.session_state["risk_page"]
        current_page = max(1, min(current_page, total_pages))  # clamp
        start_idx    = (current_page - 1) * PAGE_SIZE
        end_idx      = min(start_idx + PAGE_SIZE, total_rows)
        shown        = table.iloc[start_idx:end_idx]

        # ── Header tabel ─────────────────────────────────────────
        h1, h2, h3, h4 = st.columns([2.5, 1.5, 1.5, 2.5])
        for col, label in zip([h1,h2,h3,h4], ["Account ID","Score","Level","Top Trigger"]):
            with col:
                st.markdown(f"<div style='font-size:11px;color:#6b7280;font-weight:500;letter-spacing:.08em;text-transform:uppercase;padding:4px 0;'>{label}</div>", unsafe_allow_html=True)
        st.markdown("<hr style='margin:0 0 4px;border:none;border-top:1px solid #2d3142;'>", unsafe_allow_html=True)

        # ── Baris data ───────────────────────────────────────────
        for _, row in shown.iterrows():
            level = row["risk_level"]
            score = row["final_risk_score"]
            color = LEVEL_COLORS.get(level, "#6b7280")
            c1, c2, c3, c4 = st.columns([2.5, 1.5, 1.5, 2.5])
            with c1:
                st.markdown(f"<div style='font-weight:500;font-size:13px;color:#e2e8f0;padding:7px 0;'>{row['account_id']}</div>", unsafe_allow_html=True)
            with c2:
                st.markdown(
                    f"<div style='padding:7px 0;'>"
                    f"<div style='font-size:13px;font-weight:500;color:{color};'>{score:.1f}/100</div>"
                    f"<div style='background:#2d3142;border-radius:4px;height:3px;margin-top:4px;'>"
                    f"<div style='background:{color};width:{score}%;height:3px;border-radius:4px;'></div></div>",
                    unsafe_allow_html=True
                )
            with c3:
                st.markdown(f"<div style='padding:7px 0;'><span class='badge-{level.lower()}'>{level}</span></div>", unsafe_allow_html=True)
            with c4:
                st.markdown(f"<div style='font-size:12px;color:#9ca3af;padding:7px 0;'>{row.get('top_triggers','-')}</div>", unsafe_allow_html=True)
            st.markdown("<hr style='margin:0;border:none;border-top:0.5px solid #2d3142;'>", unsafe_allow_html=True)

        # ── Pagination controls ──────────────────────────────────
        st.markdown("<br>", unsafe_allow_html=True)

        # Info baris
        st.markdown(
            f"<div style='text-align:center;font-size:12px;color:#6b7280;margin-bottom:10px;'>"
            f"Menampilkan <b style='color:#a5b4fc;'>{start_idx+1}–{end_idx}</b> dari "
            f"<b style='color:#a5b4fc;'>{total_rows:,}</b> akun &nbsp;·&nbsp; "
            f"Page <b style='color:#e2e8f0;'>{current_page}</b> of "
            f"<b style='color:#e2e8f0;'>{total_pages}</b>"
            f"</div>",
            unsafe_allow_html=True
        )

        # Tombol navigasi — prev | 1 2 3 4 5 ... | next
        MAX_BUTTONS = 5  # jumlah nomor halaman yang ditampilkan sekaligus

        # Hitung window nomor halaman yang ditampilkan
        half = MAX_BUTTONS // 2
        page_start = max(1, current_page - half)
        page_end   = min(total_pages, page_start + MAX_BUTTONS - 1)
        if page_end - page_start < MAX_BUTTONS - 1:
            page_start = max(1, page_end - MAX_BUTTONS + 1)

        # Buat kolom: prev + nomor + next
        num_cols   = (page_end - page_start + 1) + 2  # +2 untuk prev & next
        nav_cols   = st.columns([1] * num_cols)
        col_idx    = 0

        # Tombol ◀ Prev
        with nav_cols[col_idx]:
            disabled_prev = current_page <= 1
            if st.button("◀", key="pg_prev", disabled=disabled_prev, use_container_width=True):
                st.session_state["risk_page"] = current_page - 1
                st.rerun()
        col_idx += 1

        # Tombol nomor halaman
        for pn in range(page_start, page_end + 1):
            with nav_cols[col_idx]:
                is_active = pn == current_page
                label = f"**{pn}**" if is_active else str(pn)
                btn_style = "primary" if is_active else "secondary"
                if st.button(label, key=f"pg_{pn}", type=btn_style, use_container_width=True):
                    st.session_state["risk_page"] = pn
                    st.rerun()
            col_idx += 1

        # Tombol ▶ Next
        with nav_cols[col_idx]:
            disabled_next = current_page >= total_pages
            if st.button("▶", key="pg_next", disabled=disabled_next, use_container_width=True):
                st.session_state["risk_page"] = current_page + 1
                st.rerun()


    # ════════════════════════════════════════════════════════════
    # HALAMAN 3 — DETAIL AKUN  [GRAFIK GARIS untuk perubahan pola]
    # ════════════════════════════════════════════════════════════
    elif page == "🔍 Detail Akun":
        st.markdown("# 🔍 Detail Akun")

        opts = filtered.sort_values("final_risk_score", ascending=False)["account_id"].tolist()
        if not opts:
            st.warning("Tidak ada akun yang sesuai filter.")
            st.stop()

        sel = st.selectbox(
            "Pilih Akun", opts,
            format_func=lambda x: f"{x}  —  {risk_df[risk_df['account_id']==x]['risk_level'].values[0]}  ({risk_df[risk_df['account_id']==x]['final_risk_score'].values[0]:.1f})"
        )

        acc   = risk_df[risk_df["account_id"] == sel].iloc[0]
        level = acc["risk_level"]
        score = acc["final_risk_score"]
        color = LEVEL_COLORS.get(level, "#6b7280")

        st.divider()

        h1, h2, h3 = st.columns([3, 1, 1])
        with h1:
            st.markdown(
                f"<div style='font-size:22px;font-weight:600;color:#e2e8f0;'>{sel}</div>"
                f"<div style='font-size:13px;color:#6b7280;margin-top:4px;'>Profil: {acc['profile']} · {acc.get('n_transactions','-')} transaksi</div>",
                unsafe_allow_html=True
            )
        with h2:
            st.markdown(f"<div class='metric-card'><div class='metric-label'>Risk Score</div><div class='metric-value' style='color:{color};'>{score:.1f}</div><div class='metric-sub'>dari 100</div></div>", unsafe_allow_html=True)
        with h3:
            st.markdown(f"<div class='metric-card'><div class='metric-label'>Risk Level</div><div class='metric-value' style='color:{color};font-size:22px;'>{level}</div></div>", unsafe_allow_html=True)

        st.markdown("<br>", unsafe_allow_html=True)

        # ════════════════════════════════════════════════════════
        # [BARU] GRAFIK GARIS — POLA PERUBAHAN PERILAKU
        # Semua 4 metrik utama ditampilkan sebagai line chart
        # dengan area fill untuk menunjukkan tren perubahan
        # ════════════════════════════════════════════════════════
        if features_df is not None:
            acc_f = features_df[features_df["account_id"] == sel].sort_values("day")

            if len(acc_f) > 0:
                st.markdown("<div class='section-title'>📈 Grafik Pola Perubahan Perilaku (Line Chart)</div>", unsafe_allow_html=True)

                fig_line = make_subplots(
                    rows=4, cols=1,
                    shared_xaxes=True,
                    subplot_titles=(
                        "Frekuensi Transaksi / 24 Jam",
                        "Night Ratio — Proporsi Transaksi Tengah Malam (7 hari)",
                        "Temporal Shift — Pergeseran Waktu Transaksi",
                        "Burst Score — Lonjakan Aktivitas Mendadak"
                    ),
                    vertical_spacing=0.07,
                    row_heights=[0.25, 0.25, 0.25, 0.25]
                )

                days = acc_f["day"].values

                # ── Chart 1: Frekuensi Transaksi ───────────────
                y1 = acc_f["tx_count_24h"].values
                fig_line.add_trace(
                    go.Scatter(
                        x=days, y=y1, mode="lines+markers",
                        name="Frekuensi",
                        line=dict(color="#fb923c", width=2.5, shape="spline"),
                        marker=dict(size=5, color="#fb923c"),
                        fill="tozeroy", fillcolor="rgba(251,146,60,0.08)"
                    ), row=1, col=1
                )
                # Garis rata-rata
                avg1 = np.mean(y1)
                fig_line.add_hline(y=avg1, line_color="#6b7280", line_dash="dot",
                                   line_width=1, row=1, col=1,
                                   annotation_text=f"avg {avg1:.1f}",
                                   annotation_font_color="#6b7280", annotation_font_size=10)

                # ── Chart 2: Night Ratio ────────────────────────
                y2 = acc_f["night_ratio_7d"].values
                fig_line.add_trace(
                    go.Scatter(
                        x=days, y=y2, mode="lines+markers",
                        name="Night Ratio",
                        line=dict(color="#a78bfa", width=2.5, shape="spline"),
                        marker=dict(size=5, color="#a78bfa"),
                        fill="tozeroy", fillcolor="rgba(167,139,250,0.08)"
                    ), row=2, col=1
                )
                # Threshold 30% = zona waspada
                fig_line.add_hline(y=0.30, line_color="#fcd34d", line_dash="dash",
                                   line_width=1, row=2, col=1,
                                   annotation_text="⚠ 30% threshold",
                                   annotation_font_color="#fcd34d", annotation_font_size=10)

                # ── Chart 3: Temporal Shift (LINE, bukan bar) ──
                y3 = acc_f["temporal_shift"].values
                # Pisah warna positif vs negatif dengan dua trace
                y3_pos = np.where(y3 >= 0, y3, np.nan)
                y3_neg = np.where(y3 < 0,  y3, np.nan)

                fig_line.add_trace(
                    go.Scatter(
                        x=days, y=y3, mode="lines+markers",
                        name="Temporal Shift",
                        line=dict(color="#f87171", width=2.5, shape="spline"),
                        marker=dict(
                            size=7,
                            color=["#f87171" if v >= 0 else "#6ee7b7" for v in y3],
                            line=dict(width=1, color="#1a1d27")
                        ),
                        fill="tozeroy", fillcolor="rgba(248,113,113,0.06)"
                    ), row=3, col=1
                )
                fig_line.add_hline(y=0, line_color="#6b7280", line_dash="solid",
                                   line_width=1, row=3, col=1)

                # ── Chart 4: Burst Score ────────────────────────
                y4 = acc_f["burst_score"].values if "burst_score" in acc_f.columns else np.zeros(len(days))
                fig_line.add_trace(
                    go.Scatter(
                        x=days, y=y4, mode="lines+markers",
                        name="Burst Score",
                        line=dict(color="#34d399", width=2.5, shape="spline"),
                        marker=dict(size=5, color="#34d399"),
                        fill="tozeroy", fillcolor="rgba(52,211,153,0.08)"
                    ), row=4, col=1
                )
                # Threshold burst > 2 = waspada
                fig_line.add_hline(y=2.0, line_color="#fcd34d", line_dash="dash",
                                   line_width=1, row=4, col=1,
                                   annotation_text="⚠ burst threshold",
                                   annotation_font_color="#fcd34d", annotation_font_size=10)

                # ── Layout global ──────────────────────────────
                fig_line.update_layout(
                    paper_bgcolor="rgba(0,0,0,0)",
                    plot_bgcolor="rgba(0,0,0,0)",
                    height=620,
                    showlegend=False,
                    font=dict(color="#9ca3af", size=11),
                    margin=dict(t=40, b=10, l=10, r=10),
                    hovermode="x unified"
                )
                for i in range(1, 5):
                    fig_line.update_xaxes(gridcolor="#2d3142", row=i, col=1, showgrid=True)
                    fig_line.update_yaxes(gridcolor="#2d3142", row=i, col=1, showgrid=True)

                # Label sumbu Y per row
                fig_line.update_yaxes(title_text="Tx/24h",       row=1, col=1, title_font_size=10)
                fig_line.update_yaxes(title_text="Ratio",        row=2, col=1, title_font_size=10)
                fig_line.update_yaxes(title_text="Shift",        row=3, col=1, title_font_size=10)
                fig_line.update_yaxes(title_text="Burst",        row=4, col=1, title_font_size=10)

                st.plotly_chart(fig_line, use_container_width=True)

                # Mini summary perubahan
                if len(y1) >= 2:
                    delta_freq  = y1[-1]  - y1[0]
                    delta_night = y2[-1]  - y2[0]
                    delta_shift = y3[-1]  - y3[0]

                    st.markdown("<div class='section-title'>📊 Ringkasan Perubahan Selama Periode</div>", unsafe_allow_html=True)
                    ms1, ms2, ms3 = st.columns(3)
                    with ms1:
                        arrow = "⬆️" if delta_freq > 0 else "⬇️"
                        c = "#f87171" if delta_freq > 0 else "#6ee7b7"
                        st.markdown(f"<div class='metric-card'><div class='metric-label'>Δ Frekuensi</div><div class='metric-value' style='color:{c};font-size:22px;'>{arrow} {abs(delta_freq):.1f}</div><div class='metric-sub'>tx/24h dari awal</div></div>", unsafe_allow_html=True)
                    with ms2:
                        arrow = "⬆️" if delta_night > 0 else "⬇️"
                        c = "#f87171" if delta_night > 0 else "#6ee7b7"
                        st.markdown(f"<div class='metric-card'><div class='metric-label'>Δ Night Ratio</div><div class='metric-value' style='color:{c};font-size:22px;'>{arrow} {abs(delta_night):.1%}</div><div class='metric-sub'>perubahan proporsi malam</div></div>", unsafe_allow_html=True)
                    with ms3:
                        arrow = "⬆️" if delta_shift > 0 else "⬇️"
                        c = "#f87171" if delta_shift > 0 else "#6ee7b7"
                        st.markdown(f"<div class='metric-card'><div class='metric-label'>Δ Temporal Shift</div><div class='metric-value' style='color:{c};font-size:22px;'>{arrow} {abs(delta_shift):.4f}</div><div class='metric-sub'>pergeseran waktu transaksi</div></div>", unsafe_allow_html=True)

            else:
                st.info("Data timeline tidak tersedia untuk akun ini. Grafik pola perubahan tidak dapat ditampilkan.")
        else:
            st.info("File features tidak ditemukan. Upload judolguard_features.csv untuk melihat grafik pola.")

        # ── AI Report ───────────────────────────────────────────
        st.markdown("<div class='section-title'>🤖 AI Risk Analysis (Azure OpenAI GPT-4o)</div>", unsafe_allow_html=True)
        col_btn, _ = st.columns([1, 3])
        with col_btn:
            generate_btn = st.button("✨ Generate AI Report", type="primary", use_container_width=True)

        key = f"ai_report_{sel}"
        if key not in st.session_state:
            st.session_state[key] = None

        if generate_btn:
            with st.spinner("Azure OpenAI sedang menganalisis pola perilaku..."):
                result = get_azure_recommendation(acc)
                st.session_state[key] = result if result else acc.get("explanation", "Penjelasan tidak tersedia.")

        if st.session_state[key]:
            st.markdown(f"<div class='explanation-box'>{st.session_state[key]}</div>", unsafe_allow_html=True)
        else:
            exp = acc.get("explanation", "")
            if exp and exp not in ["Klik Generate AI Report untuk analisis.", "Penjelasan belum di-generate.", ""]:
                st.markdown(f"<div class='explanation-box'>{exp}</div>", unsafe_allow_html=True)
            else:
                st.markdown("<div style='background:#1a1d27;border:1px dashed #2d3142;border-radius:8px;padding:1.5rem;text-align:center;color:#6b7280;font-size:13px;'>Klik tombol di atas untuk generate analisis dari Azure OpenAI GPT-4o</div>", unsafe_allow_html=True)

        # ── Behavioral Features ─────────────────────────────────
        st.markdown("<div class='section-title'>📈 Behavioral Features (Snapshot)</div>", unsafe_allow_html=True)
        feat_map = {
            "avg_night_ratio"   : ("Night Ratio 7d",   "%", 100),
            "avg_tx_24h"        : ("Frekuensi Tx/24h", "x",  1),
            "avg_unique_recv"   : ("Penerima Unik/7d",  "",  1),
            "avg_burst_score"   : ("Burst Score",       "x",  1),
            "avg_temporal_shift": ("Temporal Shift",     "",  1),
            "avg_qris_ratio"    : ("QRIS Ratio",        "%", 100),
        }
        cols = st.columns(3)
        for i, (k, (label, unit, mult)) in enumerate(feat_map.items()):
            if k in acc:
                with cols[i % 3]:
                    st.metric(label, f"{acc[k]*mult:.1f}{unit}")

        # ── Rekomendasi Tindakan ────────────────────────────────
        st.divider()
        st.markdown("<div class='section-title'>⚡ Rekomendasi Tindakan</div>", unsafe_allow_html=True)
        title, desc, bg, fg = REC_DETAIL.get(level, REC_DETAIL["Low"])
        st.markdown(
            f"<div style='background:{bg};border:1px solid {fg}33;border-radius:10px;padding:1.2rem 1.5rem;'>"
            f"<div style='font-size:16px;font-weight:600;color:{fg};margin-bottom:8px;'>{title}</div>"
            f"<div style='font-size:14px;color:#d1d5db;line-height:1.6;'>{desc}</div></div>",
            unsafe_allow_html=True
        )
    '''

with open('06_dashboard.py', 'w') as f:
    f.write(DASHBOARD_CODE)
print("✓ 06_dashboard.py berhasil ditulis")


✓ 06_dashboard.py berhasil ditulis
